# Fast exact FES reconstruction: interchangeable backends

This notebook uses **synthetic coordinates and weights**, not a physical BiVO4 trajectory. It compares the unchanged local backend with the optional exact separable backend and with the explicitly approximate binned backend. Only `method=` changes; arbitrary CV arrays and original bias-derived log weights still work.

For the real-COLVAR / ESTK comparison, read `docs/estk_review.md`. ESTK is not a dependency of this notebook or package.

In [1]:
from pathlib import Path
import sys, time
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root / "src"))
import numpy as np
import matplotlib.pyplot as plt
from bivo4_fes import compute_fes

rng = np.random.default_rng(42)
cv = rng.normal(0.0, 0.4, size=(2000, 2))
logweights = rng.normal(0.0, 2.0, size=len(cv))
settings = dict(bounds=[[-1.2, 1.2], [-1.2, 1.2]], bins=81,
                bandwidth=[0.08, 0.08], temperature=350, unit="eV")

/opt/pyvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Same inputs, different numerical engine

A warm-up excludes first-time Numba compilation. Runtime depends on bandwidth, grid, BLAS thread settings, and sample count. `separable` and `exact` are untruncated KDEs; `local` truncates extremely distant tails; `binned` approximates sample locations.

In [2]:
compute_fes(cv[:2], logweights[:2], method="local", **settings)
surfaces, seconds = {}, {}
for method in ("exact", "local", "separable", "binned"):
    start = time.perf_counter()
    surfaces[method] = compute_fes(cv, logweights, method=method, **settings)
    seconds[method] = time.perf_counter() - start
    print(f"{method:10s}: {seconds[method]:.4f} s")

exact     : 0.9099 s
local     : 0.1450 s
separable : 0.0058 s
binned    : 0.0037 s


In [3]:
reference = surfaces["exact"].free_energy
mask = reference < 0.4
for method, fes in surfaces.items():
    error = np.max(np.abs(fes.free_energy[mask] - reference[mask]))
    print(f"{method:10s}: max error below 0.4 eV = {error:.3e} eV")

np.testing.assert_allclose(surfaces["separable"].free_energy, reference, atol=1e-11)
np.testing.assert_allclose(surfaces["separable"].local_ess,
                           surfaces["exact"].local_ess, rtol=1e-10)
print("Exact-backend checks passed.")

exact     : max error below 0.4 eV = 0.000e+00 eV
local     : max error below 0.4 eV = 1.152e-15 eV
separable : max error below 0.4 eV = 1.665e-16 eV
binned    : max error below 0.4 eV = 1.621e-02 eV
Exact-backend checks passed.


In [4]:
fig, ax = plt.subplots()
x = surfaces["exact"].axes[0]
j = len(surfaces["exact"].axes[1]) // 2
for method in ("exact", "separable", "binned"):
    ax.plot(x, surfaces[method].free_energy[:, j], label=method)
ax.set(xlabel="Synthetic CV 1", ylabel="Free energy (eV)", title="Slice through the same synthetic KDE")
ax.legend()
plt.show()

/tmp/ipykernel_3243/3428294385.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Production use

In notebook 02, add `method="separable"` to your existing `compute_fes` call. Keep the same atom selections, state masks, frame-to-COLVAR alignment, temperature, bandwidths, and original bias-derived log weights.

The speed test does not establish convergence of any physical FES. Do not pool independently shifted biases from the four MLP simulations. Analyze each run first, then apply an explicitly defined reference-aligned model average. Changing the KDE engine does not change that statistical requirement.